# 03 · Custom Validation: Field Validators, Model Validators & Computed Fields

**Module 15 – Pydantic: Data Validation & Settings** · Estimated time: 55 min

### 🎯 What you will learn
- `@field_validator` with modes **after** (default), **before**, **wrap**, **plain**
- Validating several fields with one validator; accessing other fields via `ValidationInfo`
- `@model_validator` (`before` / `after`): cross-field rules
- ✅ Reusable validators with `Annotated[..., AfterValidator(...)]` / `BeforeValidator` / `WrapValidator` / `PlainValidator`
- Raising errors: `ValueError`, `AssertionError`, and custom `PydanticCustomError`
- `@computed_field`: derived values included in output
- `validate_default`, and the order validators run in

In [1]:
from typing import Annotated, Any, Self
from datetime import date
from pydantic import (BaseModel, Field, ValidationError, ValidationInfo, field_validator, model_validator,
                      computed_field, AfterValidator, BeforeValidator, WrapValidator, PlainValidator)
from pydantic_core import PydanticCustomError

def show(model, **data):
    try:
        print("✅", model(**data))
    except ValidationError as e:
        for err in e.errors():
            print("❌", ".".join(map(str, err["loc"])) or "(model)", "→", err["msg"])

## 1. `@field_validator`: after mode (the default)

Runs **after** Pydantic's own type validation, so the value already has the right type:

In [2]:
class User(BaseModel):
    username: str
    email: str
    age: int

    @field_validator("username")
    @classmethod
    def no_spaces(cls, v: str) -> str:
        if " " in v:
            raise ValueError("must not contain spaces")
        return v.lower()                         # validators can also TRANSFORM the value

    @field_validator("email")
    @classmethod
    def company_email(cls, v: str) -> str:
        if not v.endswith("@example.com"):
            raise ValueError("must be a company (@example.com) address")
        return v

    @field_validator("age")
    @classmethod
    def adult(cls, v: int) -> int:
        assert v >= 18, "must be 18 or older"     # AssertionError also works (but is skipped under python -O!)
        return v

show(User, username="Asha_R", email="asha@example.com", age="31")
show(User, username="asha r", email="asha@gmail.com", age=12)

✅ username='asha_r' email='asha@example.com' age=31
❌ username → Value error, must not contain spaces
❌ email → Value error, must be a company (@example.com) address
❌ age → Assertion failed, must be 18 or older


✅ Always **return** the value (possibly modified). A validator that forgets `return` sets the field to `None`!

## 2. Before mode: clean raw input first

Runs **before** type validation, so the value can be anything:

In [3]:
class Order(BaseModel):
    tags: list[str]
    amount: float

    @field_validator("tags", mode="before")
    @classmethod
    def split_csv(cls, v: Any) -> Any:
        if isinstance(v, str):                   # accept "a, b, c" as well as ["a", "b", "c"]
            return [t.strip() for t in v.split(",") if t.strip()]
        return v

    @field_validator("amount", mode="before")
    @classmethod
    def strip_currency(cls, v: Any) -> Any:
        if isinstance(v, str):
            return v.replace("₹", "").replace(",", "").strip()
        return v

print(Order(tags="urgent, gift ,", amount="₹1,250.50"))
print(Order(tags=["a"], amount=3))

tags=['urgent', 'gift'] amount=1250.5
tags=['a'] amount=3.0


## 3. Wrap and plain modes

- **wrap**: you get the value **and** a `handler` that runs the normal validation, so you can act before *and* after, or catch errors
- **plain**: replaces Pydantic's validation completely (you're responsible for the type!)

In [4]:
class Measurement(BaseModel):
    value: float
    unit: str

    @field_validator("value", mode="wrap")
    @classmethod
    def tolerant(cls, v: Any, handler) -> float:
        if v in ("", "N/A", None):
            return float("nan")                  # map missing markers before validation
        try:
            return handler(v)                    # normal float validation
        except ValidationError:
            raise PydanticCustomError("bad_reading", "reading '{reading}' is not a number", {"reading": v})

    @field_validator("unit", mode="plain")
    @classmethod
    def any_to_lower(cls, v: Any) -> str:
        return str(v).lower()                    # no type check at all

print(Measurement(value="N/A", unit="KG"), Measurement(value="3.5", unit=12))
show(Measurement, value="heavy", unit="kg")

value=nan unit='kg' value=3.5 unit='12'
❌ value → reading 'heavy' is not a number


## 4. One validator, several fields; reading other fields

In [5]:
class Registration(BaseModel):
    first_name: str
    last_name: str
    password: str
    password_confirm: str

    @field_validator("first_name", "last_name")            # several fields
    @classmethod
    def capitalise(cls, v: str) -> str:
        return v.strip().title()

    @field_validator("password_confirm")
    @classmethod
    def passwords_match(cls, v: str, info: ValidationInfo) -> str:
        # info.data holds the fields validated SO FAR (fields are validated in definition order)
        if "password" in info.data and v != info.data["password"]:
            raise ValueError("passwords do not match")
        return v

show(Registration, first_name=" asha ", last_name="rao", password="s3cret", password_confirm="s3cret")
show(Registration, first_name="ravi", last_name="k", password="a", password_confirm="b")

✅ first_name='Asha' last_name='Rao' password='s3cret' password_confirm='s3cret'
❌ password_confirm → Value error, passwords do not match


> 💡 For rules involving several fields, a **model validator** (next) is usually clearer than relying on field order.

## 5. `@model_validator`: rules about the whole model

In [6]:
class DateRange(BaseModel):
    start: date
    end: date
    max_days: int = 30

    @model_validator(mode="after")                 # runs on the fully built model
    def check_range(self) -> Self:
        if self.end < self.start:
            raise ValueError("end must be on or after start")
        if (self.end - self.start).days > self.max_days:
            raise ValueError(f"range longer than {self.max_days} days")
        return self                                # ✅ must return self

    @model_validator(mode="before")                # runs on the RAW input (often a dict)
    @classmethod
    def accept_single_day(cls, data: Any) -> Any:
        if isinstance(data, dict) and "day" in data:       # {"day": x} → {"start": x, "end": x}
            data = {**data, "start": data["day"], "end": data["day"]}
        return data

show(DateRange, start="2026-10-01", end="2026-10-20")
show(DateRange, day="2026-12-25")
show(DateRange, start="2026-10-20", end="2026-10-01")
show(DateRange, start="2026-01-01", end="2026-12-31")

✅ start=datetime.date(2026, 10, 1) end=datetime.date(2026, 10, 20) max_days=30
✅ start=datetime.date(2026, 12, 25) end=datetime.date(2026, 12, 25) max_days=30
❌ (model) → Value error, end must be on or after start
❌ (model) → Value error, range longer than 30 days


## 6. ✅ Reusable validators with `Annotated`

Attach validation to a **type**, then reuse it in any model, function or `TypeAdapter`:

In [7]:
def check_even(v: int) -> int:
    if v % 2:
        raise ValueError(f"{v} is not even")
    return v

def strip_and_lower(v: Any) -> Any:
    return v.strip().lower() if isinstance(v, str) else v

EvenInt = Annotated[int, AfterValidator(check_even)]
Slug = Annotated[str, BeforeValidator(strip_and_lower), Field(pattern=r"^[a-z0-9-]+$")]
Indian_Pincode = Annotated[str, Field(pattern=r"^[1-9][0-9]{5}$")]

class Article(BaseModel):
    slug: Slug
    pages: EvenInt
    related: list[Slug] = []
    pincode: Indian_Pincode | None = None

show(Article, slug="  Hello-World ", pages=12, related=[" PY-tips "])
show(Article, slug="bad slug!", pages=7, pincode="012345")

✅ slug='hello-world' pages=12 related=['py-tips'] pincode=None
❌ slug → String should match pattern '^[a-z0-9-]+$'
❌ pages → Value error, 7 is not even
❌ pincode → String should match pattern '^[1-9][0-9]{5}$'


Order with several validators in `Annotated`: **before**/**wrap**/**plain** validators run right-to-left (outermost first), then the core type validation, then **after** validators left-to-right.

## 7. Custom error types and messages

In [8]:
def strong_password(v: str) -> str:
    problems = []
    if len(v) < 10: problems.append("at least 10 characters")
    if not any(c.isdigit() for c in v): problems.append("a digit")
    if not any(c.isupper() for c in v): problems.append("an uppercase letter")
    if problems:
        raise PydanticCustomError("weak_password", "Password needs {needs}", {"needs": ", ".join(problems)})
    return v

class Login(BaseModel):
    password: Annotated[str, AfterValidator(strong_password)]

try:
    Login(password="short")
except ValidationError as e:
    err = e.errors()[0]
    print(err["type"], "|", err["msg"], "|", err["ctx"])

weak_password | Password needs at least 10 characters, a digit, an uppercase letter | {'needs': 'at least 10 characters, a digit, an uppercase letter'}


> 💡 Message templates use plain `{name}` placeholders filled from the context dict (no `!r` or format specs).

A stable error **`type`** (`"weak_password"`) lets API clients and translations handle errors reliably.

## 8. `@computed_field`: derived values

Properties are normally *not* included in `model_dump()`; `@computed_field` makes them part of the output (and JSON Schema):

In [9]:
class Rectangle(BaseModel):
    width: float = Field(gt=0)
    height: float = Field(gt=0)

    @computed_field
    @property
    def area(self) -> float:
        return self.width * self.height

    @computed_field
    @property
    def is_square(self) -> bool:
        return self.width == self.height

r = Rectangle(width=3, height=4)
print(r.area, r.model_dump(), r.model_dump_json())

12.0 {'width': 3.0, 'height': 4.0, 'area': 12.0, 'is_square': False} {"width":3.0,"height":4.0,"area":12.0,"is_square":false}


## 9. Validating defaults & assignment

In [10]:
class Settings(BaseModel):
    workers: EvenInt = Field(default=3, validate_default=True)    # defaults are NOT validated unless asked

try:
    Settings()
except ValidationError as e:
    print("default rejected:", e.errors()[0]["msg"])

default rejected: Value error, 3 is not even


Validators normally run only at creation. Use `model_config = ConfigDict(validate_assignment=True)` to validate on `obj.field = value` too (lesson 06).

## 📝 Summary
- `@field_validator` (after/before/wrap/plain) for single-field rules; always return the value.
- `@model_validator(mode="after")` for cross-field rules (return `self`); `mode="before"` to reshape raw input.
- ✅ Prefer reusable `Annotated[T, AfterValidator(f)]` types.
- `PydanticCustomError` gives custom error types/messages; `@computed_field` adds derived output fields.

## 🏋️ Practice
1. `Coupon` model: `code` (uppercase, letters/digits, 6–10 chars), `percent_off` (1–90), `valid_from`/`valid_to` dates with a model validator ensuring `valid_to > valid_from`.
2. A `BeforeValidator` that converts `"yes"/"no"/"y"/"n"` (any case) to booleans for a `Survey` model.
3. A `computed_field` `full_name` on a `Person(first, last)` model, and check it appears in `model_dump_json()`.